# Metro-ASR — Fine-Tuning Guide

Adapt the released Metro-Small checkpoint to your own data — a new domain, dialect, or
vocabulary — without training from scratch, and *measure* whether it actually helped.

This notebook is meant to be run top to bottom on **your own dataset** (HuggingFace or
local files) and covers the full loop:

1. Load your dataset and set aside a held-out test set
2. Evaluate the base (pretrained) model on that test set — your baseline
3. **Strategy 1** — fine-tune the acoustic model only, tokenizer untouched
4. **Strategy 2** — fine-tune *and* adapt the tokenizer to your domain/dialect
5. *(Optional)* train a new domain language-model head and compare greedy vs.
   beam search with the shipped general LM vs. beam search with your new one
6. Test the final model interactively

Every strategy is measured the same way: evaluate on the held-out test set, compare
WER/CER against the baseline, then eyeball a random sample side by side. Training is
monitored live in [Weights & Biases](https://wandb.ai).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammedAly22/metro-asr/blob/main/examples/fine_tuning.ipynb)

> **Requires a GPU.** Colab menu: *Runtime → Change runtime type → T4 GPU*.
> The language-head section is the exception — it's CPU-only and takes minutes, not hours.

## Setup

In [ ]:
import numpy
_COLAB_NUMPY = numpy.__version__
print(f"Colab's pre-installed numpy: {_COLAB_NUMPY} (pinning back to this after install)")


In [ ]:
!pip install -q "metro-asr[train,lm]"
!pip install -q "numpy=={_COLAB_NUMPY}"   # see note below
!git clone -q https://github.com/MohammedAly22/metro-asr.git
%cd metro-asr


> [!NOTE]
> `pyctcdecode` (needed for beam search) still declares `numpy<2.0.0` in its own
> package metadata, even though it runs fine under numpy 2.x — pip will honor that and
> downgrade numpy to satisfy it. On Colab, which has numpy 2.x pre-installed and several
> other packages (jax, opencv, scipy, **pyarrow** — which `datasets` depends on) already
> built against that exact version, that downgrade breaks those packages' compiled
> extensions with errors like `numpy.dtype size changed` or `cannot import name
> '_center' from 'numpy._core.umath'` the next time anything imports them. Reinstalling
> *some* numpy 2.x isn't enough to fix this — it has to be the **exact** version Colab's
> other packages were compiled against, not just the latest release, so the cell above
> captures Colab's original version *before* installing anything and pins back to that
> precise version afterward.
>
> If you already hit a numpy-related crash before adding this fix, restart the
> runtime first (*Runtime → Restart session*) — otherwise the capture cell records
> the already-broken version instead of Colab's original one.

## 1. Configuration

Everything dataset-specific lives here — the rest of the notebook doesn't need editing.

- `DATASET_ID`: a HuggingFace dataset id (`"username/dataset-name"`). Leave it as-is and
  jump to the **"Using your own local audio instead?"** cell in Section 3 if your data
  isn't on the Hub.
- `AUDIO_COL` / `TEXT_COL`: the column names in that dataset holding the audio and the
  transcript. `scripts/finetune.py` and `scripts/evaluate.py` also take these as CLI flags
  (`--audio-col` / `--text-col`, or the underscore spelling `--audio_col` / `--text_col` —
  both work) if you'd rather run them outside this notebook.

In [ ]:
MODEL_SIZE = "small"                          # small / medium / large
DATASET_ID = "your-username/your-dataset"     # HuggingFace dataset id
AUDIO_COL = "audio"                           # audio column name in DATASET_ID
TEXT_COL = "text"                             # transcript column name in DATASET_ID

# Shared fine-tuning hyperparameters — both strategies below use the same ones so the
# comparison isolates "what did adapting the tokenizer change", not "did I also change lr".
LEARNING_RATE = 5e-5
MAX_STEPS = 30000
FREEZE_STEPS = 3000

CKPT_DIR = lambda suffix: f"checkpoints/metro-{MODEL_SIZE}-{suffix}"

print(f"Dataset:  {DATASET_ID}  (audio={AUDIO_COL!r}, text={TEXT_COL!r})")
print(f"Model:    metro-{MODEL_SIZE}")


## 2. Download the pretrained checkpoint

Fine-tuning starts from the released weights, so get them first — this is the step it's
easiest to forget, and every fine-tuning command below depends on `checkpoints/` existing.
About 900 MB (weights + tokenizer). The 6 GB general-purpose language model is *not*
included here — it's only needed for the optional LM comparison in Section 7, which
downloads it itself.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=f"MohammedAly22/metro-asr-{MODEL_SIZE}",
    local_dir="checkpoints",
    allow_patterns=["config.yaml", "model.pt", "bpe.model", "bpe.vocab"],
)
print("Downloaded to checkpoints/")


## 3. Load your dataset and prepare a held-out test set

This cell:
1. Checks whether `DATASET_ID` already ships a `test` / `validation` split.
2. **If it does**, that split becomes the held-out test set, untouched by fine-tuning —
   `scripts/finetune.py` is pointed at the *other* split by name (`--split`), so there's no
   overlap between what gets trained on and what gets evaluated on.
3. **If it doesn't**, 10% of the data is carved off right here, saved once, and never
   touched again. The remaining 90% is handed to `finetune.py` via `--prepared-data`
   instead of `--dataset` — if we instead re-downloaded "the train split" by name inside
   `finetune.py`, it would include the rows we just set aside as test, silently leaking
   test data into training.

Either way it also dumps the *training* portion's transcripts to
`corpora/domain_corpus.txt` — plain text, one sentence per line — which Sections 6 and 7
reuse as the fine-tuning dataset's own ground truth for tokenizer / language-model
training.

In [ ]:
import os
from datasets import get_dataset_split_names
from metro_asr.utils.config import load_config
from metro_asr.data.dataset import load_hf_datasets

config = load_config(f"configs/metro_{MODEL_SIZE}.yaml")

try:
    available_splits = get_dataset_split_names(DATASET_ID)
except Exception:
    available_splits = ["train"]
print("Available splits:", available_splits)

TEST_SPLIT_CANDIDATES = ["test", "validation", "val", "eval"]
existing_test_split = next((s for s in TEST_SPLIT_CANDIDATES if s in available_splits), None)
train_split_name = next((s for s in available_splits if s != existing_test_split), available_splits[0])

os.makedirs("data_prepared", exist_ok=True)
os.makedirs("corpora", exist_ok=True)


def dump_corpus(ds, path):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for item in ds:
            t = item["text"].strip()
            if t:
                f.write(t.replace("\n", " ") + "\n")
    print(f"  wrote {len(ds)} lines -> {path}")


if existing_test_split:
    print(f"Found an existing '{existing_test_split}' split — using it as the held-out test set.")
    test_ds = load_hf_datasets([DATASET_ID], config, audio_col=AUDIO_COL, text_col=TEXT_COL, split=existing_test_split)
    test_ds.save_to_disk("data_prepared/test")

    train_ds = load_hf_datasets([DATASET_ID], config, audio_col=AUDIO_COL, text_col=TEXT_COL, split=train_split_name)
    dump_corpus(train_ds, "corpora/domain_corpus.txt")

    FINETUNE_ARGS = ["--dataset", DATASET_ID, "--audio-col", AUDIO_COL, "--text-col", TEXT_COL, "--split", train_split_name]
else:
    print("No test/validation split found — carving one out ourselves (10%), saved once, never touched again.")
    full_ds = load_hf_datasets([DATASET_ID], config, audio_col=AUDIO_COL, text_col=TEXT_COL, split=train_split_name)
    carved = full_ds.train_test_split(test_size=0.1, seed=42)
    carved["test"].save_to_disk("data_prepared/test")
    dump_corpus(carved["train"], "corpora/domain_corpus.txt")

    # Mirror finetune.py's own internal 95/5 train/eval split so the 10% test set above is
    # never re-downloaded or re-mixed back in — hand the 90% off via --prepared-data.
    inner = carved["train"].train_test_split(test_size=0.05, seed=42)
    inner["train"].save_to_disk("data_prepared_ft/train")
    inner["test"].save_to_disk("data_prepared_ft/eval")
    FINETUNE_ARGS = ["--prepared-data", "data_prepared_ft"]

print("\nFine-tuning will use:", " ".join(FINETUNE_ARGS))


> **Using your own local audio instead of a HuggingFace dataset?** Run this cell
> *instead of* the one above, then continue with Section 4 as normal — it produces the
> same `data_prepared/test`, `data_prepared_ft/`, `corpora/domain_corpus.txt`, and
> `FINETUNE_ARGS` that the rest of the notebook expects.

In [ ]:
from datasets import Dataset, Audio

# Replace with your own (path, transcript) pairs — dozens of examples is a lot better than
# a handful, but this format works at any size.
local_data = [
    {"audio": "path/to/audio1.wav", "text": "أنا رايح الـ meeting"},
    {"audio": "path/to/audio2.wav", "text": "الـ project ده محتاج update"},
]

local_ds = Dataset.from_list(local_data).cast_column("audio", Audio(sampling_rate=16000))
carved = local_ds.train_test_split(test_size=0.1, seed=42)
carved["test"].save_to_disk("data_prepared/test")
dump_corpus(carved["train"], "corpora/domain_corpus.txt")

inner = carved["train"].train_test_split(test_size=0.05, seed=42)
inner["train"].save_to_disk("data_prepared_ft/train")
inner["test"].save_to_disk("data_prepared_ft/eval")
FINETUNE_ARGS = ["--prepared-data", "data_prepared_ft"]

print("\nFine-tuning will use:", " ".join(FINETUNE_ARGS))


## 4. Baseline: evaluate the pretrained model

Before touching any weights, see where the released model stands on *your* data — every
fine-tuning claim below is measured against this number.

In [ ]:
!python scripts/evaluate.py \
    --config checkpoints/config.yaml \
    --checkpoint checkpoints/model.pt \
    --tokenizer-dir checkpoints \
    --name "Base (pretrained)" \
    --test-data data_prepared/test \
    --no-beam \
    --output-dir eval_results \
    --json-out eval_results/baseline.json


In [ ]:
import json


def load_wer(json_path, method="greedy"):
    """Read a scripts/evaluate.py --json-out file. Falls back to whatever method is
    present if the requested one wasn't computed (e.g. --no-beam was passed)."""
    data = json.load(open(json_path, encoding="utf-8"))
    model_name = next(iter(data))
    metrics = data[model_name].get(method)
    if metrics is None:
        method, metrics = next(iter(data[model_name].items()))
    return metrics["wer_all"], metrics["cer_all"], model_name


baseline_wer, baseline_cer, _ = load_wer("eval_results/baseline.json")
print(f"Baseline — WER: {baseline_wer:.2%}   CER: {baseline_cer:.2%}")


## 5. Strategy 1 — fine-tune the acoustic model only

The tokenizer is left exactly as released — only the encoder and CTC head adapt. This is
the right default when your data is the *same* language/dialect mix as the base model
(Egyptian Arabic + code-switched English) and you're mainly adapting to a new domain,
accent, or recording condition.

`scripts/finetune.py` freezes the encoder for the first `FREEZE_STEPS` steps so the CTC
head can adjust to the new data before the acoustic representations start moving — see the
README's [Fine-tuning](https://github.com/MohammedAly22/metro-asr#fine-tuning) section for
why. Progress (loss, learning rate, gradient norm, eval WER/CER) streams to
[Weights & Biases](https://wandb.ai) under the project/run name set in
`configs/metro_{MODEL_SIZE}.yaml` (`training.wandb_project` / `wandb_run_name`), with
`-strategy1-acoustic` appended to the run name.

Run `!wandb login` first if you want it synced to your account — skip it and W&B logs
locally instead (`wandb: offline`).

In [ ]:
!wandb login


In [ ]:
!python scripts/finetune.py \
    --config configs/metro_{MODEL_SIZE}.yaml \
    --tokenizer-dir checkpoints \
    --checkpoint checkpoints/model.pt \
    {" ".join(FINETUNE_ARGS)} \
    --lr {LEARNING_RATE} \
    --max-steps {MAX_STEPS} \
    --freeze-steps {FREEZE_STEPS} \
    --run-suffix strategy1-acoustic


In [ ]:
!python scripts/evaluate.py \
    --config configs/metro_{MODEL_SIZE}.yaml \
    --checkpoint {CKPT_DIR("strategy1-acoustic")}/best_model.pt \
    --tokenizer-dir checkpoints \
    --name "Strategy 1 (acoustic fine-tune)" \
    --test-data data_prepared/test \
    --no-beam \
    --output-dir eval_results \
    --json-out eval_results/strategy1.json


In [ ]:
strategy1_wer, strategy1_cer, _ = load_wer("eval_results/strategy1.json")
print(f"Baseline    — WER: {baseline_wer:.2%}   CER: {baseline_cer:.2%}")
print(f"Strategy 1  — WER: {strategy1_wer:.2%}   CER: {strategy1_cer:.2%}")
print(f"Change      — WER: {strategy1_wer - baseline_wer:+.2%}   CER: {strategy1_cer - baseline_cer:+.2%}")


### Compare a random sample: base vs. fine-tuned

In [ ]:
import random
from datasets import load_from_disk
from metro_asr import MetroASREngine

test_ds = load_from_disk("data_prepared/test")
sample = test_ds[random.randrange(len(test_ds))]
audio_input = (sample["audio"]["array"], sample["audio"]["sampling_rate"])

base_engine = MetroASREngine.from_local(
    config_path="checkpoints/config.yaml",
    checkpoint_path="checkpoints/model.pt",
    tokenizer_dir="checkpoints",
)
strategy1_engine = MetroASREngine.from_local(
    config_path=f"configs/metro_{MODEL_SIZE}.yaml",
    checkpoint_path=f'{CKPT_DIR("strategy1-acoustic")}/best_model.pt',
    tokenizer_dir="checkpoints",
)

print("Ground truth:    ", sample["text"])
print("Base model:      ", base_engine.transcribe(audio_input).text)
print("Strategy 1 model:", strategy1_engine.transcribe(audio_input).text)


## 6. Strategy 2 — fine-tune *and* adapt the tokenizer

Same fine-tuning loop, but starting from a fresh BPE tokenizer trained on
`corpora/domain_corpus.txt` (the fine-tuning dataset's own transcripts) instead of the
released one. This matters most when your data leans into a different dialect or a more
MSA-heavy register than Egyptian Arabic + code-switching — the released tokenizer's vocab
may simply not segment your text well, no matter how long you fine-tune the acoustic model.

The encoder and subsampling weights still transfer from the pretrained checkpoint — only
the CTC head (vocab-sized) doesn't, since the vocabulary itself changed shape. `finetune.py`
detects that automatically: it compares every checkpoint tensor's shape against the new
model's, skips the ones that don't match, and lets those (and only those) train from
scratch alongside the usual encoder freeze/unfreeze schedule.

In [ ]:
!python scripts/train_bpe_tokenizer.py \
    --corpus corpora/domain_corpus.txt \
    --vocab-size 5000 \
    --out tokenizer_strategy2


In [ ]:
!python scripts/finetune.py \
    --config configs/metro_{MODEL_SIZE}.yaml \
    --tokenizer-dir tokenizer_strategy2 \
    --checkpoint checkpoints/model.pt \
    {" ".join(FINETUNE_ARGS)} \
    --lr {LEARNING_RATE} \
    --max-steps {MAX_STEPS} \
    --freeze-steps {FREEZE_STEPS} \
    --run-suffix strategy2-tokenizer


In [ ]:
!python scripts/evaluate.py \
    --config configs/metro_{MODEL_SIZE}.yaml \
    --checkpoint {CKPT_DIR("strategy2-tokenizer")}/best_model.pt \
    --tokenizer-dir tokenizer_strategy2 \
    --name "Strategy 2 (tokenizer-adapted fine-tune)" \
    --test-data data_prepared/test \
    --no-beam \
    --output-dir eval_results \
    --json-out eval_results/strategy2.json


In [ ]:
strategy2_wer, strategy2_cer, _ = load_wer("eval_results/strategy2.json")
print(f"Baseline    — WER: {baseline_wer:.2%}   CER: {baseline_cer:.2%}")
print(f"Strategy 1  — WER: {strategy1_wer:.2%}   CER: {strategy1_cer:.2%}")
print(f"Strategy 2  — WER: {strategy2_wer:.2%}   CER: {strategy2_cer:.2%}")


### Compare a random sample: base vs. tokenizer-adapted fine-tune

In [ ]:
sample = test_ds[random.randrange(len(test_ds))]
audio_input = (sample["audio"]["array"], sample["audio"]["sampling_rate"])

strategy2_engine = MetroASREngine.from_local(
    config_path=f"configs/metro_{MODEL_SIZE}.yaml",
    checkpoint_path=f'{CKPT_DIR("strategy2-tokenizer")}/best_model.pt',
    tokenizer_dir="tokenizer_strategy2",
)

print("Ground truth:    ", sample["text"])
print("Base model:      ", base_engine.transcribe(audio_input).text)
print("Strategy 2 model:", strategy2_engine.transcribe(audio_input).text)


## 7. Optional: train a new domain language-model head

The n-gram language head is detachable and trains from **text alone** — minutes, CPU only,
no audio. Two ways to get a corpus:

- Your own text similar to the fine-tuning domain (documentation, product names, a larger
  written corpus in the same register) — point `--corpus` at that file instead.
- Reuse `corpora/domain_corpus.txt`, built from the fine-tuning dataset's own ground-truth
  transcripts in Section 3. That's what's used below.

Pick which fine-tuned model to pair it with — defaults to Strategy 1's:

In [ ]:
FINAL_CHECKPOINT = f'{CKPT_DIR("strategy1-acoustic")}/best_model.pt'
FINAL_TOKENIZER_DIR = "checkpoints"

# To use the tokenizer-adapted model instead:
# FINAL_CHECKPOINT = f'{CKPT_DIR("strategy2-tokenizer")}/best_model.pt'
# FINAL_TOKENIZER_DIR = "tokenizer_strategy2"

print(f"Checkpoint: {FINAL_CHECKPOINT}")
print(f"Tokenizer:  {FINAL_TOKENIZER_DIR}")


In [ ]:
!python scripts/train_lm.py \
    --corpus corpora/domain_corpus.txt \
    --out lm/domain_4gram.arpa \
    --order 4


Download the shipped general-purpose LM (~6 GB) to compare against:

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=f"MohammedAly22/metro-asr-{MODEL_SIZE}",
    local_dir="checkpoints",
    allow_patterns=["lm_5gram.bin"],
)
print("Shipped general-purpose LM ready at checkpoints/lm_5gram.bin")


In [ ]:
!python scripts/evaluate.py \
    --config configs/metro_{MODEL_SIZE}.yaml \
    --checkpoint {FINAL_CHECKPOINT} \
    --tokenizer-dir {FINAL_TOKENIZER_DIR} \
    --name "Fine-tuned + shipped general LM" \
    --test-data data_prepared/test \
    --lm-path checkpoints/lm_5gram.bin \
    --output-dir eval_results \
    --json-out eval_results/lm_shipped.json


In [ ]:
!python scripts/evaluate.py \
    --config configs/metro_{MODEL_SIZE}.yaml \
    --checkpoint {FINAL_CHECKPOINT} \
    --tokenizer-dir {FINAL_TOKENIZER_DIR} \
    --name "Fine-tuned + new domain LM" \
    --test-data data_prepared/test \
    --lm-path lm/domain_4gram.arpa \
    --output-dir eval_results \
    --json-out eval_results/lm_domain.json


In [ ]:
greedy_wer, greedy_cer, _ = load_wer("eval_results/lm_shipped.json", method="greedy")
shipped_lm_wer, shipped_lm_cer, _ = load_wer("eval_results/lm_shipped.json", method="beam_lm")
domain_lm_wer, domain_lm_cer, _ = load_wer("eval_results/lm_domain.json", method="beam_lm")

print(f"Greedy                    — WER: {greedy_wer:.2%}   CER: {greedy_cer:.2%}")
print(f"Beam + shipped general LM — WER: {shipped_lm_wer:.2%}   CER: {shipped_lm_cer:.2%}")
print(f"Beam + new domain LM      — WER: {domain_lm_wer:.2%}   CER: {domain_lm_cer:.2%}")


## 8. Test the final model interactively

In [ ]:
import torch
from metro_asr import MetroASREngine

engine = MetroASREngine.from_local(
    config_path=f"configs/metro_{MODEL_SIZE}.yaml",
    checkpoint_path=FINAL_CHECKPOINT,
    tokenizer_dir=FINAL_TOKENIZER_DIR,
    lm_path="lm/domain_4gram.arpa",   # remove this line to skip the LM / use greedy
    device="cuda" if torch.cuda.is_available() else "cpu",
)

result = engine.transcribe("audio.wav", beam_search=True)
print(f"Text: {result.text}")
print(f"RTF:  {result.rtf:.4f}")


## Fine-tuning reference

| Parameter | Recommended | Notes |
|---|---|---|
| Learning rate | `1e-4` to `5e-5` | 10-20x lower than pretraining |
| Freeze steps | 3,000 – 10,000 | Lets the CTC head adapt before the encoder moves |
| Max steps | 20,000 – 50,000 | Depends on dataset size — watch eval WER, stop when it turns |
| Batch size | 16 – 32 | Set in the config's `training.batch_size` |
| SpecAugment | keep enabled | Helps most on small datasets |
| Tokenizer vocab size | 3,000 – 8,000 | Smaller for narrow domains, larger for broad/multi-dialect corpora |

**When to adapt the tokenizer (Strategy 2) instead of just the acoustic model (Strategy
1):** your data is a different dialect or a much more MSA-heavy register than Egyptian
Arabic + code-switching, or the domain corpus check above shows a lot of `❌` splits on
words that matter to you. Otherwise Strategy 1 is simpler, needs less data, and keeps
compatibility with the shipped general-purpose language model.

## Next steps

- Not fine-tuning on the whole label set? Try a data subset first (`--split "train[:5000]"`)
  to sanity-check the pipeline before committing hours of compute.
- Compare more than two strategies by repeating Section 5/6's pattern with different
  `--run-suffix` values and hyperparameters — each gets its own `eval_results/*.json`.
- Once you're happy with a checkpoint, see the main
  [README](https://github.com/MohammedAly22/metro-asr#readme) for packaging it for
  `MetroASREngine.from_pretrained()` or pushing it to the Hub.